In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'c:\Users\Pradeep S\Desktop\Semester VII\Business Analytics\Case_Study\data\raw_apps.csv')
df

,app_id,title,category,price,is_free,has_in_app_purchases,contains_ads,content_rating,size_available,size_mb,...,num_ratings,num_reviews,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,developer_name,released_date,last_updated_date
0,com.adsk.sketchbook,Sketchbook,Art & Design,0.00,True,True,False,Everyone,False,NaN,...,706158.0,31173.0,88061,33424,58564,95617,430459,Sketchbook,"Oct 8, 2014",2026-05-30
1,jp.ne.ibis.ibispaintx.app,ibis Paint X,Art & Design,0.00,True,True,True,Everyone,False,NaN,...,3001736.0,126549.0,178886,70453,133739,378012,2240592,ibis inc.,"Feb 27, 2014",2026-09-11
2,ar.drawing.sketch.paint.trace.draw.picture.paper,AR Drawing: Sketch & Paint,Art & Design,0.00,True,True,False,Everyone,False,NaN,...,834940.0,3072.0,117283,15923,47833,100390,553383,AR Drawing,"Jun 2, 2023",2026-08-18
3,com.brakefield.painter,Infinite Painter,Art & Design,0.00,True,True,False,Everyone,False,NaN,...,244840.0,10548.0,12787,6486,11026,32707,181816,Infinite Studio LLC,"Apr 2, 2012",2026-09-08
4,com.vblast.flipaclip,FlipaClip: Draw 2D Animation,Art & Design,0.00,True,True,True,Everyone,False,NaN,...,795694.0,52531.0,61207,24847,49616,106702,553296,Visual Blasters LLC,"Apr 2, 2012",2026-09-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12664,es.socialpoint.wordlife,Word Life - Crossword puzzle,Word,0.00,True,True,True,Everyone,False,NaN,...,194831.0,15905.0,12663,5919,14555,32944,128737,Social Point,"May 7, 2019",2026-08-25
12665,com.fillword.cross.wordmind.en,Word Crossy - A crossword game,Word,0.00,True,True,True,Everyone,False,NaN,...,292267.0,56802.0,23056,11281,25583,52454,179886,WORD CALM,"Sep 15, 2017",2026-09-21
12666,com.theworldpremium.com,World Premium,Entertainment,0.00,True,False,True,Teen,False,NaN,...,NaN,NaN,0,0,0,0,0,SRG Pvt.Ltd,"Jan 30, 2026",2026-09-17
12667,com.BeruGames.wordsearchpremium,Word Search Premium,Word,0.99,False,False,False,Everyone,False,NaN,...,19.0,1.0,0,0,0,0,19,Beru Games,"Dec 13, 2025",NaN


In [17]:
print("Shape:",df.shape)


Shape: (12669, 23)


In [18]:
print("Columns:",list(df.columns))


Columns: ['app_id', 'title', 'category', 'price', 'is_free', 'has_in_app_purchases', 'contains_ads', 'content_rating', 'size_available', 'size_mb', 'install_count', 'real_installs', 'average_rating', 'num_ratings', 'num_reviews', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'developer_name', 'released_date', 'last_updated_date']


In [19]:
uid = df["app_id"].nunique()
print("Unique app_ids:",uid)


Unique app_ids: 12669


In [20]:
print(f"\nCategories ({df['category'].nunique()}):")
print(df['category'].value_counts().to_string())


Categories (48):
category
Tools                      720
Education                  636
Productivity               532
Health & Fitness           521
Simulation                 501
Sports                     500
Finance                    463
Puzzle                     458
Entertainment              401
Business                   374
Lifestyle                  363
Casual                     334
Shopping                   327
Music & Audio              317
Strategy                   314
Role Playing               311
Personalization            307
Travel & Local             264
Board                      251
Books & Reference          250
Maps & Navigation          247
Word                       235
Educational                231
Medical                    228
Photography                223
Action                     218
Social                     215
Card                       214
Racing                     188
Arcade                     187
Food & Drink               185
Trivia      

In [21]:
print("Missing values")
print(df.isnull().sum())

Missing values
app_id                      0
title                       0
category                    0
price                      10
is_free                     0
has_in_app_purchases        0
contains_ads                0
content_rating              0
size_available              0
size_mb                 12669
install_count              11
real_installs               0
average_rating           1775
num_ratings              1775
num_reviews              1775
ratings_1                   0
ratings_2                   0
ratings_3                   0
ratings_4                   0
ratings_5                   0
developer_name              0
released_date             173
last_updated_date        1843
dtype: int64


In [22]:

print("dtypes")
print(df.dtypes)

dtypes
app_id                   object
title                    object
category                 object
price                   float64
is_free                    bool
has_in_app_purchases       bool
contains_ads               bool
content_rating           object
size_available             bool
size_mb                 float64
install_count            object
real_installs             int64
average_rating          float64
num_ratings             float64
num_reviews             float64
ratings_1                 int64
ratings_2                 int64
ratings_3                 int64
ratings_4                 int64
ratings_5                 int64
developer_name           object
released_date            object
last_updated_date        object
dtype: object


In [23]:
# Remove size_mb/size_available — confirmed unavailable from Google Play's public pages
df = df.drop(columns=['size_mb', 'size_available'])

In [24]:
# Convert install strings like "10,000,000+" into plain numbers

df['install_count_numeric'] = df['install_count'].str.replace(',', '').str.replace('+', '').astype(float)

In [25]:
# Drop the small number of rows missing price or install count

df = df.dropna(subset=['price', 'install_count_numeric'])

In [26]:
# Parse dates, fill missing update dates with release date, then compute app age since last update

df['released_date'] = pd.to_datetime(df['released_date'], errors='coerce')
df['last_updated_date'] = pd.to_datetime(df['last_updated_date'], errors='coerce')

df['last_updated_date'] = df['last_updated_date'].fillna(df['released_date'])
df = df.dropna(subset=['last_updated_date'])

df['days_since_update'] = (pd.Timestamp.now() - df['last_updated_date']).dt.days

In [27]:
# Bucket price into free/low/mid/premium tiers

def price_tier(p):
    if p == 0:
        return 'free'
    elif p <= 2:
        return 'low'
    elif p <= 10:
        return 'mid'
    else:
        return 'premium'

df['price_tier'] = df['price'].apply(price_tier)

In [28]:
# Keep only apps with enough ratings to give a reliable average

df = df[df['num_ratings'] >= 50].copy()

In [29]:
# Label an app "high engagement" if its rating beats its own category's median

category_medians = df.groupby('category')['average_rating'].transform('median')
df['high_engagement'] = (df['average_rating'] > category_medians).astype(int)